## Teste com variáveis selecionadas - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 1. Carregando e selecionando os dados
df = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Tratamento de segurança caso haja vírgulas nos números
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()

# 2. Tratamento inteligente de falhas
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# =====================================================================
# ETAPA 1: DIVISÃO CRONOLÓGICA (Mitigando Vazamento de Dados)
# =====================================================================
ponto_de_corte = int(len(df_model) * 0.8)

# Treino (7008 horas - passado)
X_train = X.iloc[:ponto_de_corte]
y_train = y.iloc[:ponto_de_corte]

# Teste (1752 horas - futuro)
X_test = X.iloc[ponto_de_corte:]
y_test = y.iloc[ponto_de_corte:]

# =====================================================================
# ETAPA 2: AVALIAÇÃO ESTATÍSTICA OLS (Baseada no Treino)
# =====================================================================
print("--- RESULTADOS ESTATÍSTICOS (OLS) ---")
X_train_sm = sm.add_constant(X_train)
model_ols = sm.OLS(y_train, X_train_sm).fit()
print(model_ols.summary())

# =====================================================================
# ETAPA 3: AVALIAÇÃO PREDITIVA RANDOM FOREST (Baseada no Teste)
# =====================================================================
print("\n--- RESULTADOS RANDOM FOREST ---")
rf = RandomForestRegressor(random_state=42, n_estimators=100)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
r2_rf = r2_score(y_test, y_pred)

print(f"R² do Random Forest em dados futuros: {r2_rf:.4f}")

importances = pd.Series(rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis no Random Forest (em % de contribuição):")
print(importances * 100)

--- RESULTADOS ESTATÍSTICOS (OLS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.718
Model:                                                      OLS   Adj. R-squared:                  0.717
Method:                                           Least Squares   F-statistic:                     2964.
Date:                                          Tue, 23 Jun 2026   Prob (F-statistic):               0.00
Time:                                                  22:15:36   Log-Likelihood:                -16495.
No. Observations:                                          7008   AIC:                         3.300e+04
Df Residuals:                                              7001   BIC:                         3.305e+04
Df Model:                                                     6                                         
Covariance Type: 

## Teste com todas as variáveis (Sem filtro) - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

# 1. Carregando os dados
df = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança (caso haja vírgulas)
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# =====================================================================
# NOVO CENÁRIO: TODAS AS 11 VARIÁVEIS (Incluindo Máx, Mín e Rajadas)
# =====================================================================
X_cols_all = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_all = df[[target_col] + X_cols_all].copy()

# Tratamento inteligente de falhas
df_model_all['RADIACAO GLOBAL (Kj/m²)'] = df_model_all['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_all = df_model_all.interpolate(method='linear').dropna()

X_all = df_model_all[X_cols_all]
y_all = df_model_all[target_col]

# 2. Divisão Cronológica
ponto_de_corte = int(len(df_model_all) * 0.8)
X_train_all = X_all.iloc[:ponto_de_corte]
y_train_all = y_all.iloc[:ponto_de_corte]
X_test_all = X_all.iloc[ponto_de_corte:]
y_test_all = y_all.iloc[ponto_de_corte:]

# =====================================================================
# AVALIAÇÃO ESTATÍSTICA (OLS)
# =====================================================================
print("--- RESULTADOS ESTATÍSTICOS OLS (TODAS AS VARIÁVEIS) ---")
X_train_sm_all = sm.add_constant(X_train_all)
model_ols_all = sm.OLS(y_train_all, X_train_sm_all).fit()
print(model_ols_all.summary())

# =====================================================================
# AVALIAÇÃO PREDITIVA (RANDOM FOREST)
# =====================================================================
print("\n--- RESULTADOS RANDOM FOREST (TODAS AS VARIÁVEIS) ---")
rf_all = RandomForestRegressor(random_state=42, n_estimators=100)
rf_all.fit(X_train_all, y_train_all)

y_pred_all = rf_all.predict(X_test_all)
r2_rf_all = r2_score(y_test_all, y_pred_all)

print(f"R² do Random Forest em dados futuros: {r2_rf_all:.4f}")

importances_all = pd.Series(rf_all.feature_importances_, index=X_cols_all).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_all * 100)

--- RESULTADOS ESTATÍSTICOS OLS (TODAS AS VARIÁVEIS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.721
Model:                                                      OLS   Adj. R-squared:                  0.721
Method:                                           Least Squares   F-statistic:                     1645.
Date:                                          Sun, 12 Apr 2026   Prob (F-statistic):               0.00
Time:                                                  01:25:28   Log-Likelihood:                -16450.
No. Observations:                                          7008   AIC:                         3.292e+04
Df Residuals:                                              6996   BIC:                         3.301e+04
Df Model:                                                    11                                        

## Teste com variáveis selecionadas + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df_a = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df_a.columns:
    if df_a[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_a[col] = pd.to_numeric(df_a[col].str.replace(',', '.'), errors='coerce')
        except:
            df_a[col] = pd.to_numeric(df_a[col], errors='coerce')

# 2. Engenharia de Atributos
df_a['Hora'] = df_a['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_a['Mes'] = pd.to_datetime(df_a['Data']).dt.month

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 3. Variáveis do Cenário A (Sem Multicolinearidade)
X_cols_selected = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_a = df_a[[target_col] + X_cols_selected].copy()
df_model_a['RADIACAO GLOBAL (Kj/m²)'] = df_model_a['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_a = df_model_a.interpolate(method='linear').dropna()

X_a = df_model_a[X_cols_selected]
y_a = df_model_a[target_col]

# 4. Divisão Cronológica
ponto_de_corte_a = int(len(df_model_a) * 0.8)
X_train_a, X_test_a = X_a.iloc[:ponto_de_corte_a], X_a.iloc[ponto_de_corte_a:]
y_train_a, y_test_a = y_a.iloc[:ponto_de_corte_a], y_a.iloc[ponto_de_corte_a:]

# 5. Avaliação
print("=======================================================")
print("CENÁRIO A: VARIÁVEIS SELECIONADAS + HORA E MÊS")
print("=======================================================\n")

print("--- RESULTADOS ESTATÍSTICOS (OLS) ---")
X_train_sm_a = sm.add_constant(X_train_a)
model_ols_a = sm.OLS(y_train_a, X_train_sm_a).fit()
print(model_ols_a.summary())

print("\n--- RESULTADOS RANDOM FOREST ---")
rf_a = RandomForestRegressor(random_state=42, n_estimators=100)
rf_a.fit(X_train_a, y_train_a)
y_pred_a = rf_a.predict(X_test_a)

print(f"R² do Random Forest em dados futuros: {r2_score(y_test_a, y_pred_a):.4f}")

importances_a = pd.Series(rf_a.feature_importances_, index=X_cols_selected).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_a * 100)

CENÁRIO A: VARIÁVEIS SELECIONADAS + HORA E MÊS

--- RESULTADOS ESTATÍSTICOS (OLS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.775
Model:                                                      OLS   Adj. R-squared:                  0.775
Method:                                           Least Squares   F-statistic:                     3021.
Date:                                          Sun, 12 Apr 2026   Prob (F-statistic):               0.00
Time:                                                  17:43:06   Log-Likelihood:                -15691.
No. Observations:                                          7008   AIC:                         3.140e+04
Df Residuals:                                              6999   BIC:                         3.146e+04
Df Model:                                                     8           

## Teste com todas as variáveis (Sem filtro) + Hora e mês - Regressão Linear (OLS) e Random Forest

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df_b = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df_b.columns:
    if df_b[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_b[col] = pd.to_numeric(df_b[col].str.replace(',', '.'), errors='coerce')
        except:
            df_b[col] = pd.to_numeric(df_b[col], errors='coerce')

# 2. Engenharia de Atributos
df_b['Hora'] = df_b['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_b['Mes'] = pd.to_datetime(df_b['Data']).dt.month

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 3. Variáveis do Cenário B (Com Redundâncias)
X_cols_all = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_b = df_b[[target_col] + X_cols_all].copy()
df_model_b['RADIACAO GLOBAL (Kj/m²)'] = df_model_b['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_b = df_model_b.interpolate(method='linear').dropna()

X_b = df_model_b[X_cols_all]
y_b = df_model_b[target_col]

# 4. Divisão Cronológica
ponto_de_corte_b = int(len(df_model_b) * 0.8)
X_train_b, X_test_b = X_b.iloc[:ponto_de_corte_b], X_b.iloc[ponto_de_corte_b:]
y_train_b, y_test_b = y_b.iloc[:ponto_de_corte_b], y_b.iloc[ponto_de_corte_b:]

# 5. Avaliação
print("=======================================================")
print("CENÁRIO B: TODAS AS VARIÁVEIS + HORA E MÊS")
print("=======================================================\n")

print("--- RESULTADOS ESTATÍSTICOS (OLS) ---")
X_train_sm_b = sm.add_constant(X_train_b)
model_ols_b = sm.OLS(y_train_b, X_train_sm_b).fit()
print(model_ols_b.summary())

print("\n--- RESULTADOS RANDOM FOREST ---")
rf_b = RandomForestRegressor(random_state=42, n_estimators=100)
rf_b.fit(X_train_b, y_train_b)
y_pred_b = rf_b.predict(X_test_b)

print(f"R² do Random Forest em dados futuros: {r2_score(y_test_b, y_pred_b):.4f}")

importances_b = pd.Series(rf_b.feature_importances_, index=X_cols_all).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_b * 100)

CENÁRIO B: TODAS AS VARIÁVEIS + HORA E MÊS

--- RESULTADOS ESTATÍSTICOS (OLS) ---
                                         OLS Regression Results                                         
Dep. Variable:     TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)   R-squared:                       0.777
Model:                                                      OLS   Adj. R-squared:                  0.777
Method:                                           Least Squares   F-statistic:                     1876.
Date:                                          Sun, 12 Apr 2026   Prob (F-statistic):               0.00
Time:                                                  18:06:39   Log-Likelihood:                -15664.
No. Observations:                                          7008   AIC:                         3.136e+04
Df Residuals:                                              6994   BIC:                         3.145e+04
Df Model:                                                    13               

## Teste com variáveis selecionadas - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis Selecionadas (O cenário puro, sem Mês e Hora)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (VARIÁVEIS SELECIONADAS)")
print("=======================================================\n")

# n_estimators=100 (100 árvores sequenciais)
# learning_rate=0.1 (tamanho do "passo" que ele dá para consertar o erro)
gb_model = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model.fit(X_train, y_train)

y_pred = gb_model.predict(X_test)
r2_gb = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb:.4f}")

importances_gb = pd.Series(gb_model.feature_importances_, index=X_cols).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb * 100)

MODELO: GRADIENT BOOSTING (VARIÁVEIS SELECIONADAS)

R² do Gradient Boosting em dados futuros: 0.7144

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      46.913076
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    35.984174
RADIACAO GLOBAL (Kj/m²)                                  14.579761
VENTO, VELOCIDADE HORARIA (m/s)                           1.800413
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.650886
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.071692
dtype: float64


## Teste com todas as variáveis (Sem filtro) - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. TODAS as 11 Variáveis (Com redundâncias de Máx e Mín, mas SEM Hora e Mês)
X_cols_all = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model = df[[target_col] + X_cols_all].copy()

# Tratamento inteligente de falhas
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols_all]
y = df_model[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_de_corte = int(len(df_model) * 0.8)
X_train, X_test = X.iloc[:ponto_de_corte], X.iloc[ponto_de_corte:]
y_train, y_test = y.iloc[:ponto_de_corte], y.iloc[ponto_de_corte:]

# =====================================================================
# MACHINE LEARNING: GRADIENT BOOSTING (TODAS AS VARIÁVEIS)
# =====================================================================
print("=======================================================")
print("MODELO: GRADIENT BOOSTING (TODAS AS 11 VARIÁVEIS)")
print("=======================================================\n")

gb_model_all = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model_all.fit(X_train, y_train)

y_pred = gb_model_all.predict(X_test)
r2_gb_all = r2_score(y_test, y_pred)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb_all:.4f}")

importances_gb_all = pd.Series(gb_model_all.feature_importances_, index=X_cols_all).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_gb_all * 100)

MODELO: GRADIENT BOOSTING (TODAS AS 11 VARIÁVEIS)

R² do Gradient Boosting em dados futuros: 0.7202

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      39.535148
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    14.008819
RADIACAO GLOBAL (Kj/m²)                                  13.715410
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         12.093436
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           9.879214
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  6.306032
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  1.689552
VENTO, RAJADA MAXIMA (m/s)                                1.320404
VENTO, VELOCIDADE HORARIA (m/s)                           0.838544
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.556679
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.056763
dtype: float64


## Teste com variáveis selecionadas + Hora e mês - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados
df_c = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

for col in df_c.columns:
    if df_c[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_c[col] = pd.to_numeric(df_c[col].str.replace(',', '.'), errors='coerce')
        except:
            df_c[col] = pd.to_numeric(df_c[col], errors='coerce')

# 2. Engenharia de Atributos (Hora e Mês)
df_c['Hora'] = df_c['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_c['Mes'] = pd.to_datetime(df_c['Data']).dt.month

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 3. Variáveis Selecionadas + Hora e Mês
X_cols_selected = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_c = df_c[[target_col] + X_cols_selected].copy()
df_model_c['RADIACAO GLOBAL (Kj/m²)'] = df_model_c['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_c = df_model_c.interpolate(method='linear').dropna()

X_c = df_model_c[X_cols_selected]
y_c = df_model_c[target_col]

# 4. Divisão Cronológica (80/20)
ponto_c = int(len(df_model_c) * 0.8)
X_train_c, X_test_c = X_c.iloc[:ponto_c], X_c.iloc[ponto_c:]
y_train_c, y_test_c = y_c.iloc[:ponto_c], y_c.iloc[ponto_c:]

print("=======================================================")
print("MODELO: GB (VARIÁVEIS SELECIONADAS + HORA E MÊS)")
print("=======================================================\n")

gb_model_c = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model_c.fit(X_train_c, y_train_c)

y_pred_c = gb_model_c.predict(X_test_c)
r2_gb_c = r2_score(y_test_c, y_pred_c)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb_c:.4f}")

importances_c = pd.Series(gb_model_c.feature_importances_, index=X_cols_selected).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_c * 100)

MODELO: GB (VARIÁVEIS SELECIONADAS + HORA E MÊS)

R² do Gradient Boosting em dados futuros: 0.7736

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      42.090305
Mes                                                      31.595313
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    13.502053
RADIACAO GLOBAL (Kj/m²)                                   8.620732
Hora                                                      2.922068
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.644830
VENTO, VELOCIDADE HORARIA (m/s)                           0.608750
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.015949
dtype: float64


## Teste com todas as variáveis (Sem filtro) + Hora e mês - Gradient Boosting

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados
df_d = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

for col in df_d.columns:
    if df_d[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_d[col] = pd.to_numeric(df_d[col].str.replace(',', '.'), errors='coerce')
        except:
            df_d[col] = pd.to_numeric(df_d[col], errors='coerce')

# 2. Engenharia de Atributos (Hora e Mês)
df_d['Hora'] = df_d['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_d['Mes'] = pd.to_datetime(df_d['Data']).dt.month

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 3. Todas as 11 Variáveis + Hora e Mês
X_cols_all = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_d = df_d[[target_col] + X_cols_all].copy()
df_model_d['RADIACAO GLOBAL (Kj/m²)'] = df_model_d['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_d = df_model_d.interpolate(method='linear').dropna()

X_d = df_model_d[X_cols_all]
y_d = df_model_d[target_col]

# 4. Divisão Cronológica (80/20)
ponto_d = int(len(df_model_d) * 0.8)
X_train_d, X_test_d = X_d.iloc[:ponto_d], X_d.iloc[ponto_d:]
y_train_d, y_test_d = y_d.iloc[:ponto_d], y_d.iloc[ponto_d:]

print("=======================================================")
print("MODELO: GB (TODAS AS 11 VARIÁVEIS + HORA E MÊS)")
print("=======================================================\n")

gb_model_d = GradientBoostingRegressor(random_state=42, n_estimators=100, learning_rate=0.1)
gb_model_d.fit(X_train_d, y_train_d)

y_pred_d = gb_model_d.predict(X_test_d)
r2_gb_d = r2_score(y_test_d, y_pred_d)

print(f"R² do Gradient Boosting em dados futuros: {r2_gb_d:.4f}")

importances_d = pd.Series(gb_model_d.feature_importances_, index=X_cols_all).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_d * 100)

MODELO: GB (TODAS AS 11 VARIÁVEIS + HORA E MÊS)

R² do Gradient Boosting em dados futuros: 0.7773

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      34.055394
Mes                                                      31.395670
RADIACAO GLOBAL (Kj/m²)                                   8.816718
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     7.768755
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  6.936924
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           3.448122
Hora                                                      2.700854
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          2.279447
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  1.026185
VENTO, RAJADA MAXIMA (m/s)                                0.777745
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.624543
VENTO, VELOCIDADE HORARIA (m/s)                           0.166554
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.003

## Teste com variáveis selecionadas - XGBoost


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor

from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df_xgb1 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df_xgb1.columns:
    if df_xgb1[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb1[col] = pd.to_numeric(df_xgb1[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb1[col] = pd.to_numeric(df_xgb1[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df_xgb1[[target_col] + X_cols_1].copy()
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear').dropna()

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica
ponto_1 = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_1], X_1.iloc[ponto_1:]
y_train_1, y_test_1 = y_1.iloc[:ponto_1], y_1.iloc[ponto_1:]

print("=======================================================")
print("MODELO: XGBOOST (VARIÁVEIS SELECIONADAS - SEM TEMPO)")
print("=======================================================\n")

# 4. Treinando o XGBoost
xgb_1 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_1.fit(X_train_1, y_train_1)

y_pred_1 = xgb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"R² do XGBoost em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(xgb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

MODELO: XGBOOST (VARIÁVEIS SELECIONADAS - SEM TEMPO)

R² do XGBoost em dados futuros: 0.7046

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      53.510826
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    26.395706
RADIACAO GLOBAL (Kj/m²)                                  12.362233
VENTO, VELOCIDADE HORARIA (m/s)                           3.858847
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      2.776776
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          1.095610
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df_xgb2 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df_xgb2.columns:
    if df_xgb2[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb2[col] = pd.to_numeric(df_xgb2[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb2[col] = pd.to_numeric(df_xgb2[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: Todas as 11 Variáveis (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_xgb2[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica
ponto_2 = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_2], X_2.iloc[ponto_2:]
y_train_2, y_test_2 = y_2.iloc[:ponto_2], y_2.iloc[ponto_2:]

print("=======================================================")
print("MODELO: XGBOOST (TODAS AS 11 VARIÁVEIS - SEM TEMPO)")
print("=======================================================\n")

# 4. Treinando o XGBoost
xgb_2 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_2.fit(X_train_2, y_train_2)

y_pred_2 = xgb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"R² do XGBoost em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(xgb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)

MODELO: XGBOOST (TODAS AS 11 VARIÁVEIS - SEM TEMPO)

R² do XGBoost em dados futuros: 0.7120

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      34.967571
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         19.911386
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 12.935061
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     8.286359
RADIACAO GLOBAL (Kj/m²)                                   7.750626
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           5.801640
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  4.828261
VENTO, RAJADA MAXIMA (m/s)                                1.918117
VENTO, VELOCIDADE HORARIA (m/s)                           1.690742
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.517476
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.392762
dtype: float32


## Teste com variáveis selecionadas + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df_xgb3 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df_xgb3.columns:
    if df_xgb3[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb3[col] = pd.to_numeric(df_xgb3[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb3[col] = pd.to_numeric(df_xgb3[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Hora e Mês)
df_xgb3['Hora'] = df_xgb3['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_xgb3['Mes'] = pd.to_datetime(df_xgb3['Data']).dt.month

# 3. CENÁRIO 3: Variáveis Selecionadas + Hora e Mês
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_xgb3[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 4. Divisão Cronológica
ponto_3 = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_3], X_3.iloc[ponto_3:]
y_train_3, y_test_3 = y_3.iloc[:ponto_3], y_3.iloc[ponto_3:]

print("=======================================================")
print("MODELO: XGBOOST (VARIÁVEIS SELECIONADAS + HORA/MÊS)")
print("=======================================================\n")

# 5. Treinando o XGBoost
xgb_3 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_3.fit(X_train_3, y_train_3)

y_pred_3 = xgb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"R² do XGBoost em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(xgb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)

MODELO: XGBOOST (VARIÁVEIS SELECIONADAS + HORA/MÊS)

R² do XGBoost em dados futuros: 0.7660

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      41.713280
Mes                                                      37.562199
RADIACAO GLOBAL (Kj/m²)                                   6.411407
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     6.113000
Hora                                                      4.694266
VENTO, VELOCIDADE HORARIA (m/s)                           1.698297
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.459687
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.347861
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados
df_xgb4 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

# Limpeza de segurança
for col in df_xgb4.columns:
    if df_xgb4[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_xgb4[col] = pd.to_numeric(df_xgb4[col].str.replace(',', '.'), errors='coerce')
        except:
            df_xgb4[col] = pd.to_numeric(df_xgb4[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Hora e Mês)
df_xgb4['Hora'] = df_xgb4['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_xgb4['Mes'] = pd.to_datetime(df_xgb4['Data']).dt.month

# 3. CENÁRIO 4: TODAS as 11 Variáveis + Hora e Mês
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_xgb4[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 4. Divisão Cronológica
ponto_4 = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_4], X_4.iloc[ponto_4:]
y_train_4, y_test_4 = y_4.iloc[:ponto_4], y_4.iloc[ponto_4:]

print("=======================================================")
print("MODELO: XGBOOST (TODAS AS VARIÁVEIS + HORA/MÊS)")
print("=======================================================\n")

# 5. Treinando o XGBoost
xgb_4 = XGBRegressor(random_state=42, n_estimators=100, learning_rate=0.1, objective='reg:squarederror')
xgb_4.fit(X_train_4, y_train_4)

y_pred_4 = xgb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"R² do XGBoost em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(xgb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)

MODELO: XGBOOST (TODAS AS VARIÁVEIS + HORA/MÊS)

R² do XGBoost em dados futuros: 0.7763

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      38.967724
Mes                                                      25.973024
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 10.367590
RADIACAO GLOBAL (Kj/m²)                                   5.110685
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     4.186436
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  3.749391
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           3.062134
Hora                                                      3.024000
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          2.056418
VENTO, RAJADA MAXIMA (m/s)                                1.259599
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.056972
VENTO, VELOCIDADE HORARIA (m/s)                           0.995202
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.190818
dtype:

## Teste com variáveis selecionadas - XGBoost tunado


In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados
df_tune1 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

for col in df_tune1.columns:
    if df_tune1[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune1[col] = pd.to_numeric(df_tune1[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune1[col] = pd.to_numeric(df_tune1[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df_tune1[[target_col] + X_cols_1].copy()
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear').dropna()

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica
ponto_1 = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_1], X_1.iloc[ponto_1:]
y_train_1, y_test_1 = y_1.iloc[:ponto_1], y_1.iloc[ponto_1:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 1 - SEM TEMPO)")
print("Isso pode levar cerca de 1 a 2 minutos...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],          # Número de árvores
    'learning_rate': [0.01, 0.05, 0.1, 0.2],  # Passo de aprendizado
    'max_depth': [3, 4, 5, 6],                # Profundidade máxima de cada árvore
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0], # % de variáveis usadas por árvore
    'subsample': [0.7, 0.8, 0.9, 1.0]         # % de dados usados por árvore
}

# Criando o modelo base
xgb_base = XGBRegressor(random_state=42, objective='reg:squarederror')

# Criando o buscador aleatório (testará 20 combinações diferentes)
random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1 # Usa todos os núcleos do processador do Colab
)

# 5. Treinando e encontrando o melhor modelo
random_search.fit(X_train_1, y_train_1)
melhor_xgb_1 = random_search.best_estimator_

# 6. Avaliação Final
y_pred_1 = melhor_xgb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"\n--- RESULTADOS DO FINE TUNING ---")
print(f"Melhores Hiperparâmetros encontrados: {random_search.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(melhor_xgb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 1 - SEM TEMPO)
Isso pode levar cerca de 1 a 2 minutos...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING ---
Melhores Hiperparâmetros encontrados: {'subsample': 0.9, 'n_estimators': 200, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.7}
R² do XGBoost Otimizado em dados futuros: 0.7122

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      34.844864
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    33.346592
RADIACAO GLOBAL (Kj/m²)                                  24.185890
VENTO, VELOCIDADE HORARIA (m/s)                           3.207720
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          2.555439
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.859496
dtype: float32


## Teste com todas as variáveis (Sem filtro) - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados
df_tune2 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

for col in df_tune2.columns:
    if df_tune2[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune2[col] = pd.to_numeric(df_tune2[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune2[col] = pd.to_numeric(df_tune2[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: TODAS AS 11 Variáveis (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_tune2[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica
ponto_2 = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_2], X_2.iloc[ponto_2:]
y_train_2, y_test_2 = y_2.iloc[:ponto_2], y_2.iloc[ponto_2:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 2 - 11 VARS)")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

xgb_base_2 = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search_2 = RandomizedSearchCV(
    estimator=xgb_base_2,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Treinando e encontrando o melhor modelo
random_search_2.fit(X_train_2, y_train_2)
melhor_xgb_2 = random_search_2.best_estimator_

# 6. Avaliação Final
y_pred_2 = melhor_xgb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 2) ---")
print(f"Melhores Hiperparâmetros: {random_search_2.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(melhor_xgb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 2 - 11 VARS)

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 2) ---
Melhores Hiperparâmetros: {'subsample': 0.8, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1, 'colsample_bytree': 0.7}
R² do XGBoost Otimizado em dados futuros: 0.7211

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      28.555256
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 17.850456
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)          13.372037
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         12.839752
RADIACAO GLOBAL (Kj/m²)                                  12.578654
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     9.902580
VENTO, RAJADA MAXIMA (m/s)                                1.358598
VENTO, VELOCIDADE HORARIA (m/s)                           1.299484
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  1.267545
VENTO, DIREÇÃO HORAR

## Teste com variáveis selecionadas + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados
df_tune3 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

for col in df_tune3.columns:
    if df_tune3[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune3[col] = pd.to_numeric(df_tune3[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune3[col] = pd.to_numeric(df_tune3[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Hora e Mês)
df_tune3['Hora'] = df_tune3['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune3['Mes'] = pd.to_datetime(df_tune3['Data']).dt.month

# 3. CENÁRIO 3: Variáveis Selecionadas + Hora e Mês
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_tune3[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 4. Divisão Cronológica
ponto_3 = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_3], X_3.iloc[ponto_3:]
y_train_3, y_test_3 = y_3.iloc[:ponto_3], y_3.iloc[ponto_3:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 3 - COM TEMPO)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 5. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
}

xgb_base_3 = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search_3 = RandomizedSearchCV(
    estimator=xgb_base_3,
    param_distributions=param_dist,
    n_iter=100,  # 100 iterações
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 6. Treinando e encontrando o melhor modelo
random_search_3.fit(X_train_3, y_train_3)
melhor_xgb_3 = random_search_3.best_estimator_

# 7. Avaliação Final
y_pred_3 = melhor_xgb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 3) ---")
print(f"Melhores Hiperparâmetros: {random_search_3.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(melhor_xgb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 3 - COM TEMPO)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 3) ---
Melhores Hiperparâmetros: {'subsample': 0.9, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.9}
R² do XGBoost Otimizado em dados futuros: 0.7473

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      33.937069
RADIACAO GLOBAL (Kj/m²)                                  25.539339
Mes                                                      18.713305
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    13.843896
Hora                                                      5.421195
VENTO, VELOCIDADE HORARIA (m/s)                           1.081601
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.057928
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.405670
dtype: float32


## Teste com todas as variáveis (Sem filtro) + Hora e mês - XGBoost tunado

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando e Limpando os Dados
df_tune4 = pd.read_excel('Mirante_Sao_Paulo_2025.xlsx')

for col in df_tune4.columns:
    if df_tune4[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_tune4[col] = pd.to_numeric(df_tune4[col].str.replace(',', '.'), errors='coerce')
        except:
            df_tune4[col] = pd.to_numeric(df_tune4[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Hora e Mês)
df_tune4['Hora'] = df_tune4['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_tune4['Mes'] = pd.to_datetime(df_tune4['Data']).dt.month

# 3. CENÁRIO 4: TODAS AS 11 Variáveis + Hora e Mês
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_tune4[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 4. Divisão Cronológica
ponto_4 = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_4], X_4.iloc[ponto_4:]
y_train_4, y_test_4 = y_4.iloc[:ponto_4], y_4.iloc[ponto_4:]

print("=======================================================")
print("INICIANDO FINE TUNING: XGBOOST (CENÁRIO 4 - TUDO)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 5. Definindo a Grade de Hiperparâmetros
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0]
}

xgb_base_4 = XGBRegressor(random_state=42, objective='reg:squarederror')

random_search_4 = RandomizedSearchCV(
    estimator=xgb_base_4,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 6. Treinando e encontrando o melhor modelo
random_search_4.fit(X_train_4, y_train_4)
melhor_xgb_4 = random_search_4.best_estimator_

# 7. Avaliação Final
y_pred_4 = melhor_xgb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"\n--- RESULTADOS DO FINE TUNING (CENÁRIO 4) ---")
print(f"Melhores Hiperparâmetros: {random_search_4.best_params_}")
print(f"R² do XGBoost Otimizado em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(melhor_xgb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)

INICIANDO FINE TUNING: XGBOOST (CENÁRIO 4 - TUDO)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (CENÁRIO 4) ---
Melhores Hiperparâmetros: {'subsample': 0.6, 'n_estimators': 300, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.6}
R² do XGBoost Otimizado em dados futuros: 0.7700

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      22.241980
Mes                                                      19.804373
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                 17.053223
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          8.236942
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  7.920752
RADIACAO GLOBAL (Kj/m²)                                   6.260420
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           6.125840
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     5.060397
Hora                                 

## Teste com variáveis selecionadas - Gradient Boosting tunado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df_mirante1 = pd.read_excel(nome_arquivo)

# Limpeza de segurança para garantir tipos numéricos
for col in df_mirante1.columns:
    if df_mirante1[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_mirante1[col] = pd.to_numeric(df_mirante1[col].str.replace(',', '.'), errors='coerce')
        except:
            df_mirante1[col] = pd.to_numeric(df_mirante1[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 1: Variáveis Selecionadas (Sem Tempo)
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model_1 = df_mirante1[[target_col] + X_cols_1].copy()
df_model_1['RADIACAO GLOBAL (Kj/m²)'] = df_model_1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_1 = df_model_1.interpolate(method='linear').dropna()

X_1 = df_model_1[X_cols_1]
y_1 = df_model_1[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_1 = int(len(df_model_1) * 0.8)
X_train_1, X_test_1 = X_1.iloc[:ponto_1], X_1.iloc[ponto_1:]
y_train_1, y_test_1 = y_1.iloc[:ponto_1], y_1.iloc[ponto_1:]

print("=======================================================")
print("FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 1)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros específica do Gradient Boosting
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'max_features': [0.6, 0.7, 0.8, 0.9, 1.0]  # Equivalente ao colsample_bytree
}

gb_base = GradientBoostingRegressor(random_state=42)

random_search_1 = RandomizedSearchCV(
    estimator=gb_base,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Executando a Otimização
random_search_1.fit(X_train_1, y_train_1)
melhor_gb_1 = random_search_1.best_estimator_

# 6. Avaliação do Modelo Otimizado
y_pred_1 = melhor_gb_1.predict(X_test_1)
r2_1 = r2_score(y_test_1, y_pred_1)

print(f"\n--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 1) ---")
print(f"Melhores Hiperparâmetros: {random_search_1.best_params_}")
print(f"R² do Gradient Boosting Otimizado em dados futuros: {r2_1:.4f}")

importances_1 = pd.Series(melhor_gb_1.feature_importances_, index=X_cols_1).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_1 * 100)

FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 1)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 1) ---
Melhores Hiperparâmetros: {'subsample': 0.6, 'n_estimators': 200, 'max_features': 1.0, 'max_depth': 3, 'learning_rate': 0.05}
R² do Gradient Boosting Otimizado em dados futuros: 0.7124

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      47.282415
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    35.842796
RADIACAO GLOBAL (Kj/m²)                                  13.505688
VENTO, VELOCIDADE HORARIA (m/s)                           2.125884
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      1.127607
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.115610
dtype: float64


## Teste com todas as variáveis (Sem filtro) - Gradient Boosting tunado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df_mirante2 = pd.read_excel(nome_arquivo)

# Limpeza de segurança para garantir tipos numéricos
for col in df_mirante2.columns:
    if df_mirante2[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_mirante2[col] = pd.to_numeric(df_mirante2[col].str.replace(',', '.'), errors='coerce')
        except:
            df_mirante2[col] = pd.to_numeric(df_mirante2[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. CENÁRIO 2: Todas as 11 Variáveis (Sem Tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df_mirante2[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Divisão Cronológica (80% Treino / 20% Teste)
ponto_2 = int(len(df_model_2) * 0.8)
X_train_2, X_test_2 = X_2.iloc[:ponto_2], X_2.iloc[ponto_2:]
y_train_2, y_test_2 = y_2.iloc[:ponto_2], y_2.iloc[ponto_2:]

print("=======================================================")
print("FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 2)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 4. Definindo a Grade de Hiperparâmetros específica do Gradient Boosting
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'max_features': [0.6, 0.7, 0.8, 0.9, 1.0]
}

gb_base_2 = GradientBoostingRegressor(random_state=42)

random_search_2 = RandomizedSearchCV(
    estimator=gb_base_2,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 5. Executando a Otimização
random_search_2.fit(X_train_2, y_train_2)
melhor_gb_2 = random_search_2.best_estimator_

# 6. Avaliação do Modelo Otimizado
y_pred_2 = melhor_gb_2.predict(X_test_2)
r2_2 = r2_score(y_test_2, y_pred_2)

print(f"\n--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 2) ---")
print(f"Melhores Hiperparâmetros: {random_search_2.best_params_}")
print(f"R² do Gradient Boosting Otimizado em dados futuros: {r2_2:.4f}")

importances_2 = pd.Series(melhor_gb_2.feature_importances_, index=X_cols_2).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_2 * 100)

FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 2)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 2) ---
Melhores Hiperparâmetros: {'subsample': 0.6, 'n_estimators': 200, 'max_features': 1.0, 'max_depth': 3, 'learning_rate': 0.05}
R² do Gradient Boosting Otimizado em dados futuros: 0.7216

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      35.642134
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    14.361500
RADIACAO GLOBAL (Kj/m²)                                  13.842361
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)         12.405278
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  9.850185
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           8.798331
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  1.478948
VENTO, RAJADA MAXIMA (m/s)                                1.477118
VENTO, VELOCIDADE 

## Teste com variáveis selecionadas + Hora e mês - Gradient Boosting tunado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df_mirante3 = pd.read_excel(nome_arquivo)

# Limpeza de segurança para garantir tipos numéricos
for col in df_mirante3.columns:
    if df_mirante3[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_mirante3[col] = pd.to_numeric(df_mirante3[col].str.replace(',', '.'), errors='coerce')
        except:
            df_mirante3[col] = pd.to_numeric(df_mirante3[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df_mirante3['Hora'] = df_mirante3['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_mirante3['Mes'] = pd.to_datetime(df_mirante3['Data']).dt.month

# 3. CENÁRIO 3: Variáveis Selecionadas + Hora e Mês
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df_mirante3[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_3 = int(len(df_model_3) * 0.8)
X_train_3, X_test_3 = X_3.iloc[:ponto_3], X_3.iloc[ponto_3:]
y_train_3, y_test_3 = y_3.iloc[:ponto_3], y_3.iloc[ponto_3:]

print("=======================================================")
print("FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 3)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 5. Definindo a Grade de Hiperparâmetros específica do Gradient Boosting
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'max_features': [0.6, 0.7, 0.8, 0.9, 1.0]
}

gb_base_3 = GradientBoostingRegressor(random_state=42)

random_search_3 = RandomizedSearchCV(
    estimator=gb_base_3,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 6. Executando a Otimização
random_search_3.fit(X_train_3, y_train_3)
melhor_gb_3 = random_search_3.best_estimator_

# 7. Avaliação do Modelo Otimizado
y_pred_3 = melhor_gb_3.predict(X_test_3)
r2_3 = r2_score(y_test_3, y_pred_3)

print(f"\n--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 3) ---")
print(f"Melhores Hiperparâmetros: {random_search_3.best_params_}")
print(f"R² do Gradient Boosting Otimizado em dados futuros: {r2_3:.4f}")

importances_3 = pd.Series(melhor_gb_3.feature_importances_, index=X_cols_3).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_3 * 100)

FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 3)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 3) ---
Melhores Hiperparâmetros: {'subsample': 0.6, 'n_estimators': 400, 'max_features': 0.7, 'max_depth': 4, 'learning_rate': 0.01}
R² do Gradient Boosting Otimizado em dados futuros: 0.7531

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      37.834255
Mes                                                      25.946007
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)    19.195815
RADIACAO GLOBAL (Kj/m²)                                  12.793418
Hora                                                      2.759982
VENTO, VELOCIDADE HORARIA (m/s)                           0.829191
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                      0.624748
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                          0.016584
dtype: float64


## Teste com todas as variáveis (Sem filtro) + Hora e mês - Gradient Boosting tunado

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import RandomizedSearchCV
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df_mirante4 = pd.read_excel(nome_arquivo)

# Limpeza de segurança para garantir tipos numéricos
for col in df_mirante4.columns:
    if df_mirante4[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df_mirante4[col] = pd.to_numeric(df_mirante4[col].str.replace(',', '.'), errors='coerce')
        except:
            df_mirante4[col] = pd.to_numeric(df_mirante4[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Engenharia de Atributos (Extraindo Hora e Mês)
df_mirante4['Hora'] = df_mirante4['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df_mirante4['Mes'] = pd.to_datetime(df_mirante4['Data']).dt.month

# 3. CENÁRIO 4: TODAS as 11 Variáveis + Hora e Mês
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df_mirante4[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 4. Divisão Cronológica (80% Treino / 20% Teste)
ponto_4 = int(len(df_model_4) * 0.8)
X_train_4, X_test_4 = X_4.iloc[:ponto_4], X_4.iloc[ponto_4:]
y_train_4, y_test_4 = y_4.iloc[:ponto_4], y_4.iloc[ponto_4:]

print("=======================================================")
print("FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 4)")
print("Isso vai levar alguns minutos (300 treinamentos)...")
print("=======================================================\n")

# 5. Definindo a Grade de Hiperparâmetros específica do Gradient Boosting
param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.6, 0.7, 0.8, 0.9, 1.0],
    'max_features': [0.6, 0.7, 0.8, 0.9, 1.0]
}

gb_base_4 = GradientBoostingRegressor(random_state=42)

random_search_4 = RandomizedSearchCV(
    estimator=gb_base_4,
    param_distributions=param_dist,
    n_iter=100,
    scoring='r2',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)

# 6. Executando a Otimização
random_search_4.fit(X_train_4, y_train_4)
melhor_gb_4 = random_search_4.best_estimator_

# 7. Avaliação do Modelo Otimizado
y_pred_4 = melhor_gb_4.predict(X_test_4)
r2_4 = r2_score(y_test_4, y_pred_4)

print(f"\n--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 4) ---")
print(f"Melhores Hiperparâmetros: {random_search_4.best_params_}")
print(f"R² do Gradient Boosting Otimizado em dados futuros: {r2_4:.4f}")

importances_4 = pd.Series(melhor_gb_4.feature_importances_, index=X_cols_4).sort_values(ascending=False)
print("\nImportância das Variáveis (em %):")
print(importances_4 * 100)

FINE TUNING: GRADIENT BOOSTING (MIRANTE - CENÁRIO 4)
Isso vai levar alguns minutos (300 treinamentos)...

Fitting 3 folds for each of 100 candidates, totalling 300 fits

--- RESULTADOS DO FINE TUNING (MIRANTE - CENÁRIO 4) ---
Melhores Hiperparâmetros: {'subsample': 0.6, 'n_estimators': 400, 'max_features': 0.7, 'max_depth': 4, 'learning_rate': 0.01}
R² do Gradient Boosting Otimizado em dados futuros: 0.7596

Importância das Variáveis (em %):
UMIDADE RELATIVA DO AR, HORARIA (%)                      33.255585
Mes                                                      28.111527
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)                  8.211885
RADIACAO GLOBAL (Kj/m²)                                   7.563290
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)     6.435714
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)          5.356391
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)           4.100969
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)                  3.048766
Hora              

## Teste com variáveis selecionadas - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 1
X_cols_1 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

# Isolando e tratando nulos
df_pca1 = df[X_cols_1].copy()
df_pca1['RADIACAO GLOBAL (Kj/m²)'] = df_pca1['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca1 = df_pca1.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_pca1)

# 4. Executando o PCA
pca = PCA(random_state=42)
pca.fit(X_scaled)

# 5. Calculando a Variância Explicada
var_explicada = pca.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: MIRANTE DE SANTANA - CENÁRIO 1 (SEM TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada[0] + var_explicada[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_1))],
    index=X_cols_1
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1 (o eixo mais importante)
loadings['Impacto_Absoluto_PC1'] = loadings['PC1'].abs()
tabela_final = loadings.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final[['PC1', 'Impacto_Absoluto_PC1']])

PCA: MIRANTE DE SANTANA - CENÁRIO 1 (SEM TEMPO)

Variância no Componente 1 (PC1): 33.71%
Variância no Componente 2 (PC2): 19.85%
Variância Acumulada (PC1 + PC2): 53.56%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
RADIACAO GLOBAL (Kj/m²)                             0.592798   
UMIDADE RELATIVA DO AR, HORARIA (%)                -0.590500   
VENTO, VELOCIDADE HORARIA (m/s)                     0.432148   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))                0.269746   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI... -0.179819   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                   -0.089723   

                                                    Impacto_Absoluto_PC1  
RADIACAO GLOBAL (Kj/m²)                                         0.592798  
UMIDADE RELATIVA DO AR, HORARIA (%)                             0.590500  
VENTO, VELOCIDADE HORARIA (m/s)                                 0.432148  
VENTO, DIREÇÃO HORA

## Teste com todas as variáveis (Sem filtro) - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Definição das variáveis do Cenário 2 (Todas as 11 - Sem tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

# Isolando e tratando nulos
df_pca2 = df[X_cols_2].copy()
df_pca2['RADIACAO GLOBAL (Kj/m²)'] = df_pca2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca2 = df_pca2.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória (Z-score) com StandardScaler
scaler = StandardScaler()
X_scaled_2 = scaler.fit_transform(df_pca2)

# 4. Executando o PCA
pca_2 = PCA(random_state=42)
pca_2.fit(X_scaled_2)

# 5. Calculando a Variância Explicada
var_explicada_2 = pca_2.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: MIRANTE DE SANTANA - CENÁRIO 2 (11 VARS - SEM TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada_2[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada_2[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada_2[0] + var_explicada_2[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings_2 = pd.DataFrame(
    pca_2.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_2))],
    index=X_cols_2
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings_2['Impacto_Absoluto_PC1'] = loadings_2['PC1'].abs()
tabela_final_2 = loadings_2.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final_2[['PC1', 'Impacto_Absoluto_PC1']])

PCA: MIRANTE DE SANTANA - CENÁRIO 2 (11 VARS - SEM TEMPO)

Variância no Componente 1 (PC1): 36.70%
Variância no Componente 2 (PC2): 26.10%
Variância Acumulada (PC1 + PC2): 62.80%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.435428   
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.431777   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.426486   
RADIACAO GLOBAL (Kj/m²)                            -0.353846   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.267621   
VENTO, RAJADA MAXIMA (m/s)                         -0.256570   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.234778   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.228447   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.224376   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.145294   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                    0.

## Teste com variáveis selecionadas + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# Engenharia de Atributos para o Cenário 3 (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 2. Definição das variáveis do Cenário 3
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

# Isolando e tratando nulos
df_pca3 = df[X_cols_3].copy()
df_pca3['RADIACAO GLOBAL (Kj/m²)'] = df_pca3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca3 = df_pca3.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória com StandardScaler
scaler = StandardScaler()
X_scaled_3 = scaler.fit_transform(df_pca3)

# 4. Executando o PCA
pca_3 = PCA(random_state=42)
pca_3.fit(X_scaled_3)

# 5. Calculando a Variância Explicada
var_explicada_3 = pca_3.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: MIRANTE DE SANTANA - CENÁRIO 3 (SELECIONADAS + TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada_3[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada_3[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada_3[0] + var_explicada_3[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings_3 = pd.DataFrame(
    pca_3.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_3))],
    index=X_cols_3
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings_3['Impacto_Absoluto_PC1'] = loadings_3['PC1'].abs()
tabela_final_3 = loadings_3.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final_3[['PC1', 'Impacto_Absoluto_PC1']])

PCA: MIRANTE DE SANTANA - CENÁRIO 3 (SELECIONADAS + TEMPO)

Variância no Componente 1 (PC1): 29.34%
Variância no Componente 2 (PC2): 14.90%
Variância Acumulada (PC1 + PC2): 44.24%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.537081   
RADIACAO GLOBAL (Kj/m²)                            -0.527969   
Hora                                               -0.430211   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.388447   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.247051   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.151265   
Mes                                                -0.097663   
PRECIPITAÇÃO TOTAL, HORÁRIO (mm)                    0.058024   

                                                    Impacto_Absoluto_PC1  
UMIDADE RELATIVA DO AR, HORARIA (%)                             0.537081  
RADIACAO GLOBAL (Kj/m²)       

## Teste com todas as variáveis (Sem filtro) + Hora e mês - PCA

In [ ]:
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana (Padrão Hardcoded)
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza e conversão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

# Engenharia de Atributos (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 2. Definição das variáveis do Cenário 4 (Todas as 11 + Hora e Mês)
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

# Isolando e tratando nulos
df_pca4 = df[X_cols_4].copy()
df_pca4['RADIACAO GLOBAL (Kj/m²)'] = df_pca4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_pca4 = df_pca4.interpolate(method='linear').dropna()

# 3. Padronização Obrigatória (Z-score)
scaler = StandardScaler()
X_scaled_4 = scaler.fit_transform(df_pca4)

# 4. Executando o PCA
pca_4 = PCA(random_state=42)
pca_4.fit(X_scaled_4)

# 5. Calculando a Variância Explicada
var_explicada_4 = pca_4.explained_variance_ratio_ * 100

print("=======================================================")
print("PCA: MIRANTE DE SANTANA - CENÁRIO 4 (TODAS + TEMPO)")
print("=======================================================\n")
print(f"Variância no Componente 1 (PC1): {var_explicada_4[0]:.2f}%")
print(f"Variância no Componente 2 (PC2): {var_explicada_4[1]:.2f}%")
print(f"Variância Acumulada (PC1 + PC2): {var_explicada_4[0] + var_explicada_4[1]:.2f}%\n")

# 6. Criando a Tabela de Cargas (Loadings) para entender o impacto
loadings_4 = pd.DataFrame(
    pca_4.components_.T,
    columns=[f'PC{i+1}' for i in range(len(X_cols_4))],
    index=X_cols_4
)

# Adiciona coluna de impacto absoluto para ordenar pelo PC1
loadings_4['Impacto_Absoluto_PC1'] = loadings_4['PC1'].abs()
tabela_final_4 = loadings_4.sort_values(by='Impacto_Absoluto_PC1', ascending=False)

print("Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):")
print(tabela_final_4[['PC1', 'Impacto_Absoluto_PC1']])

PCA: MIRANTE DE SANTANA - CENÁRIO 4 (TODAS + TEMPO)

Variância no Componente 1 (PC1): 33.15%
Variância no Componente 2 (PC2): 22.25%
Variância Acumulada (PC1 + PC2): 55.40%

Matriz de Cargas no PC1 (Ordenado por Relevância Estrutural):
                                                         PC1  \
UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)            0.427423   
UMIDADE RELATIVA DO AR, HORARIA (%)                 0.421385   
UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)            0.415907   
RADIACAO GLOBAL (Kj/m²)                            -0.347404   
Hora                                               -0.270527   
VENTO, VELOCIDADE HORARIA (m/s)                    -0.261845   
VENTO, RAJADA MAXIMA (m/s)                         -0.253618   
PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARI...  0.200782   
PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)    0.195612   
PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)     0.191104   
VENTO, DIREÇÃO HORARIA (gr) (° (gr))               -0.143622

## Teste com variáveis selecionadas - Gradient Boosting (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança para garantir dados numéricos
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 1 (Física pura, sem variáveis de tempo)
X_cols = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)'
]

df_model = df[[target_col] + X_cols].copy()
df_model['RADIACAO GLOBAL (Kj/m²)'] = df_model['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model = df_model.interpolate(method='linear').dropna()

X = df_model[X_cols]
y = df_model[target_col]

# 3. Configurando o modelo com os Hiperparâmetros Otimizados do seu Tuning (C1)
modelo_tunado = GradientBoostingRegressor(
    subsample=0.6,
    n_estimators=200,
    max_features=1.0,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle)")
print("Foco: Estabilidade Estrutural das Regras Físicas")
print("Aviso: Risco de Data Leakage por interpolação de dados vizinhos")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado, X, y, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV)")
print("Foco: Capacidade Real de Previsão do Tempo (Simulação de Futuro)")
print("Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X)):
    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    modelo_tunado.fit(X_train, y_train)
    score = modelo_tunado.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle)
Foco: Estabilidade Estrutural das Regras Físicas
Aviso: Risco de Data Leakage por interpolação de dados vizinhos
Dobra Aleatória 1: R² = 0.7746
Dobra Aleatória 2: R² = 0.7643
Dobra Aleatória 3: R² = 0.7710
Dobra Aleatória 4: R² = 0.7752
Dobra Aleatória 5: R² = 0.7687

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.7708
Desvio Padrão das Dobras: 0.0040

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV)
Foco: Capacidade Real de Previsão do Tempo (Simulação de Futuro)
Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.3779
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = -0.5266
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.3370
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7862
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6826

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.3314
Desvio Padrão Temporal: 0.46

## Teste com todas as variáveis (Sem filtro) - Gradient Boosting (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança para garantir dados numéricos
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# 2. Variáveis do Cenário 2 (Todas as 11 variáveis físicas, sem tempo)
X_cols_2 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)'
]

df_model_2 = df[[target_col] + X_cols_2].copy()
df_model_2['RADIACAO GLOBAL (Kj/m²)'] = df_model_2['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_2 = df_model_2.interpolate(method='linear').dropna()

X_2 = df_model_2[X_cols_2]
y_2 = df_model_2[target_col]

# 3. Configurando com os Hiperparâmetros Otimizados do Tuning (C2)
modelo_tunado_2 = GradientBoostingRegressor(
    subsample=0.6,
    n_estimators=200,
    max_features=1.0,
    max_depth=3,
    learning_rate=0.05,
    random_state=42
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - CENÁRIO 2")
print("Foco: Estabilidade Estrutural das Regras Físicas (11 Variáveis)")
print("Aviso: Risco de Data Leakage por interpolação de dados vizinhos")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado_2, X_2, y_2, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - CENÁRIO 2")
print("Foco: Capacidade Real de Previsão do Tempo (Simulação de Futuro)")
print("Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_2)):
    X_train, X_test = X_2.iloc[train_index], X_2.iloc[test_index]
    y_train, y_test = y_2.iloc[train_index], y_2.iloc[test_index]

    modelo_tunado_2.fit(X_train, y_train)
    score = modelo_tunado_2.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - CENÁRIO 2
Foco: Estabilidade Estrutural das Regras Físicas (11 Variáveis)
Aviso: Risco de Data Leakage por interpolação de dados vizinhos
Dobra Aleatória 1: R² = 0.7803
Dobra Aleatória 2: R² = 0.7728
Dobra Aleatória 3: R² = 0.7775
Dobra Aleatória 4: R² = 0.7831
Dobra Aleatória 5: R² = 0.7707

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.7769
Desvio Padrão das Dobras: 0.0046

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - CENÁRIO 2
Foco: Capacidade Real de Previsão do Tempo (Simulação de Futuro)
Vantagem: Totalmente livre de Data Leakage (Proibido espiar o futuro)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.3963
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = -0.5687
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.3516
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7886
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.6898

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Re

## Teste com variáveis selecionadas + Hora e mês - Gradient Boosting (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# Engenharia de Atributos para o Cenário 3 (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 2. Variáveis do Cenário 3 (Físicas Selecionadas + Tempo)
X_cols_3 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'Hora',
    'Mes'
]

df_model_3 = df[[target_col] + X_cols_3].copy()
df_model_3['RADIACAO GLOBAL (Kj/m²)'] = df_model_3['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_3 = df_model_3.interpolate(method='linear').dropna()

X_3 = df_model_3[X_cols_3]
y_3 = df_model_3[target_col]

# 3. Configurando com os Hiperparâmetros Otimizados do Tuning do Cenário 3
modelo_tunado_3 = GradientBoostingRegressor(
    subsample=0.6,
    n_estimators=400,
    max_features=0.7,
    max_depth=4,
    learning_rate=0.01,
    random_state=42
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - CENÁRIO 3")
print("Foco: Estabilidade Estrutural com inclusão de Hora e Mês")
print("Aviso: Alto risco de Data Leakage por interpolação temporal")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado_3, X_3, y_3, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - CENÁRIO 3")
print("Foco: Capacidade Real de Previsão do Tempo com variáveis temporais")
print("Vantagem: Livre de Data Leakage (O modelo agora tem calendário)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_3)):
    X_train, X_test = X_3.iloc[train_index], X_3.iloc[test_index]
    y_train, y_test = y_3.iloc[train_index], y_3.iloc[test_index]

    modelo_tunado_3.fit(X_train, y_train)
    score = modelo_tunado_3.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - CENÁRIO 3
Foco: Estabilidade Estrutural com inclusão de Hora e Mês
Aviso: Alto risco de Data Leakage por interpolação temporal
Dobra Aleatória 1: R² = 0.8811
Dobra Aleatória 2: R² = 0.8746
Dobra Aleatória 3: R² = 0.8704
Dobra Aleatória 4: R² = 0.8737
Dobra Aleatória 5: R² = 0.8626

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.8725
Desvio Padrão das Dobras: 0.0060

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - CENÁRIO 3
Foco: Capacidade Real de Previsão do Tempo com variáveis temporais
Vantagem: Livre de Data Leakage (O modelo agora tem calendário)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.3496
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.2420
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.6346
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7248
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7044

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.5311
Desvi

## Teste com todas as variáveis (Sem filtro) + Hora e mês - Gradient Boosting (Cross-Validation)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit
import warnings

warnings.filterwarnings('ignore')

# 1. Carregando os Dados do Mirante de Santana
nome_arquivo = 'Mirante_Sao_Paulo_2025.xlsx'
df = pd.read_excel(nome_arquivo)

# Limpeza padrão de segurança
for col in df.columns:
    if df[col].dtype == 'object' and col not in ['Data', 'Hora UTC']:
        try:
            df[col] = pd.to_numeric(df[col].str.replace(',', '.'), errors='coerce')
        except:
            df[col] = pd.to_numeric(df[col], errors='coerce')

target_col = 'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)'

# Engenharia de Atributos para o Cenário 4 (Extraindo Hora e Mês)
df['Hora'] = df['Hora UTC'].str.replace(' UTC', '').astype(int) / 100
df['Mes'] = pd.to_datetime(df['Data']).dt.month

# 2. Variáveis do Cenário 4 (Todas as 11 + Hora e Mês)
X_cols_4 = [
    'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
    'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
    'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
    'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
    'RADIACAO GLOBAL (Kj/m²)',
    'UMIDADE RELATIVA DO AR, HORARIA (%)',
    'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
    'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
    'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
    'VENTO, VELOCIDADE HORARIA (m/s)',
    'VENTO, RAJADA MAXIMA (m/s)',
    'Hora',
    'Mes'
]

df_model_4 = df[[target_col] + X_cols_4].copy()
df_model_4['RADIACAO GLOBAL (Kj/m²)'] = df_model_4['RADIACAO GLOBAL (Kj/m²)'].fillna(0)
df_model_4 = df_model_4.interpolate(method='linear').dropna()

X_4 = df_model_4[X_cols_4]
y_4 = df_model_4[target_col]

# 3. Configurando com os Hiperparâmetros Otimizados do Tuning do Cenário 4
modelo_tunado_4 = GradientBoostingRegressor(
    subsample=0.6,
    n_estimators=400,
    max_features=0.7,
    max_depth=4,
    learning_rate=0.01,
    random_state=42
)

# =====================================================================
# ABORDAGEM A: K-FOLD TRADICIONAL (VALIDAÇÃO ESTRUTURAL / SHUFFLE TRUE)
# =====================================================================
print("=====================================================================")
print("MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - CENÁRIO 4")
print("Foco: Estabilidade Estrutural com 11 Variáveis + Tempo")
print("Aviso: Alto risco de Data Leakage por interpolação temporal")
print("=====================================================================")

cv_kfold = KFold(n_splits=5, shuffle=True, random_state=42)
scores_kfold = cross_val_score(modelo_tunado_4, X_4, y_4, cv=cv_kfold, scoring='r2', n_jobs=-1)

for i, score in enumerate(scores_kfold):
    print(f"Dobra Aleatória {i+1}: R² = {score:.4f}")

print("\n--- RESUMO MÉTODO A (K-FOLD) ---")
print(f"R² Médio Global: {np.mean(scores_kfold):.4f}")
print(f"Desvio Padrão das Dobras: {np.std(scores_kfold):.4f}\n")


# =====================================================================
# ABORDAGEM B: TIME SERIES SPLIT (VALIDAÇÃO TEMPORAL REAL / SEM SHUFFLE)
# =====================================================================
print("=====================================================================")
print("MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - CENÁRIO 4")
print("Foco: Capacidade Real de Previsão do Tempo (Cenário Caótico)")
print("Vantagem: Livre de Data Leakage (Todas as físicas + Calendário)")
print("=====================================================================")

tscv = TimeSeriesSplit(n_splits=5)
scores_temporal = []

for fold, (train_index, test_index) in enumerate(tscv.split(X_4)):
    X_train, X_test = X_4.iloc[train_index], X_4.iloc[test_index]
    y_train, y_test = y_4.iloc[train_index], y_4.iloc[test_index]

    modelo_tunado_4.fit(X_train, y_train)
    score = modelo_tunado_4.score(X_test, y_test)
    scores_temporal.append(score)
    print(f"Dobra Temporal {fold+1} (Treino: {len(X_train)}h | Teste: {len(X_test)}h): R² = {score:.4f}")

print("\n--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---")
print(f"R² Médio Global Real: {np.mean(scores_temporal):.4f}")
print(f"Desvio Padrão Temporal: {np.std(scores_temporal):.4f}")
print("=====================================================================")

MÉTODO A: K-FOLD TRADICIONAL (5-FOLD CV com Shuffle) - CENÁRIO 4
Foco: Estabilidade Estrutural com 11 Variáveis + Tempo
Aviso: Alto risco de Data Leakage por interpolação temporal
Dobra Aleatória 1: R² = 0.8823
Dobra Aleatória 2: R² = 0.8775
Dobra Aleatória 3: R² = 0.8739
Dobra Aleatória 4: R² = 0.8753
Dobra Aleatória 5: R² = 0.8635

--- RESUMO MÉTODO A (K-FOLD) ---
R² Médio Global: 0.8745
Desvio Padrão das Dobras: 0.0062

MÉTODO B: TIME SERIES SPLIT (5-SPLIT TS-CV) - CENÁRIO 4
Foco: Capacidade Real de Previsão do Tempo (Cenário Caótico)
Vantagem: Livre de Data Leakage (Todas as físicas + Calendário)
Dobra Temporal 1 (Treino: 1460h | Teste: 1460h): R² = 0.3846
Dobra Temporal 2 (Treino: 2920h | Teste: 1460h): R² = 0.1734
Dobra Temporal 3 (Treino: 4380h | Teste: 1460h): R² = 0.5976
Dobra Temporal 4 (Treino: 5840h | Teste: 1460h): R² = 0.7117
Dobra Temporal 5 (Treino: 7300h | Teste: 1460h): R² = 0.7124

--- RESUMO MÉTODO B (TIME SERIES SPLIT) ---
R² Médio Global Real: 0.5159
Desvio Padrão